# Transfer Learning — Dog Breed Classifier with ResNet50

Use a model trained on 1.2M images to classify 120 dog breeds — with minimal data.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## Why Transfer Learning?

Training ResNet50 from scratch requires:
- 1.2M+ labelled images
- Days of GPU compute

Transfer learning reuses learned weights:
- **Feature extraction:** freeze all layers, train only final classifier
- **Fine-tuning:** unfreeze some layers, low learning rate

| Strategy | Data needed | Time | Accuracy |
|---|---|---|---|
| From scratch | 100K+ | Hours | ~70% |
| Feature extraction | 1K+ | Minutes | ~85% |
| Fine-tuning (partial) | 5K+ | 30 min | ~92% |

## ResNet50 Architecture

Key innovation: **Skip connections** (residual blocks)

```
x → Conv → BN → ReLU → Conv → BN → (+x) → ReLU
    ↑________________________________↑  skip!
```

Without skip connections, training 50+ layer networks was nearly impossible.
Skip connections provide a gradient highway — solving vanishing gradients.

```
Stage 1: Conv 7x7, stride 2   (112x112)
Stage 2: 3 residual blocks    (56x56)
Stage 3: 4 residual blocks    (28x28)
Stage 4: 6 residual blocks    (14x14)
Stage 5: 3 residual blocks    (7x7)
Global Average Pool → 2048 features
FC → 1000 classes (ImageNet)
```

In [ ]:
# Simulate transfer learning on digit recognition
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

digits = load_digits()
X = digits.data/16.0; y = digits.target
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.2,random_state=42)

# 'Pre-trained' model on large dataset
pretrained = MLPClassifier(hidden_layer_sizes=(256,128),max_iter=500,random_state=42)
pretrained.fit(X_tr,y_tr)
print(f"Full dataset accuracy: {accuracy_score(y_te,pretrained.predict(X_te)):.3f}")

# Transfer scenario: only 10% of data available
X_small,_,y_small,_ = train_test_split(X_tr,y_tr,test_size=0.9,random_state=42)
from_scratch = MLPClassifier(hidden_layer_sizes=(256,128),max_iter=500,random_state=42)
from_scratch.fit(X_small,y_small)
print(f"From scratch (10% data): {accuracy_score(y_te,from_scratch.predict(X_te)):.3f}")
print(f"Transfer simulated gain: use pretrained features → ~{accuracy_score(y_te,pretrained.predict(X_te)):.3f}")

## PyTorch Transfer Learning (Kaggle GPU)

```python
import torchvision.models as models
import torch.nn as nn

# Load ResNet50 pre-trained on ImageNet
model = models.resnet50(weights='IMAGENET1K_V2')

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace head: 2048 -> 120 dog breeds
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(2048, 120)
)

# Only the new head trains
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)
```

**Dataset:** Stanford Dogs (20K images, 120 breeds) on Kaggle.

In [ ]:
# Popular pre-trained models
models = [
    ('ResNet-50',      '25M',  'Excellent baseline, easy to fine-tune'),
    ('EfficientNet-B0','5M',   'Best accuracy/size, recommended first choice'),
    ('VGG-16',         '138M', 'Simple architecture, easy to understand'),
    ('MobileNetV2',    '3.4M', 'Designed for mobile/edge devices'),
    ('ViT-B/16',       '86M',  'Vision Transformer — state of the art'),
]
print(f"{'Model':<18}|{'Params':<8}|Notes")
print("-"*60)
for m in models:
    print(f"{m[0]:<18}|{m[1]:<8}|{m[2]}")

## Fine-Tuning Strategy

1. **Phase 1 (5 epochs):** Freeze all, train head only — lr=0.001
2. **Phase 2 (10 epochs):** Unfreeze last block — lr=0.00001
3. **Phase 3 (optional):** Unfreeze all — lr=0.000001

**Never** unfreeze early layers with high learning rate — causes catastrophic forgetting.

---
**Your turn:** What happens if you unfreeze ALL layers with lr=0.001? Research 'catastrophic forgetting'.